In [1]:
import os
from pathlib import Path
import pandas as pd
from fredapi import Fred

In [2]:
# Get the parent directory for the authoring of the datasets
parent_dir = Path.cwd().parent

In [3]:
# Instantiate Fred
fred = Fred(api_key_file='fred-api-key.txt') # Get API key at https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
# Get CPI Data
# According to this article (https://www.richmondfed.org/research/national_economy/macro_minute/2025/phantom_figures_missing_data_in_october) the data is missing for October 2025 due to the government shutdown
# This may impact all of the FRED data
cpi_data = fred.get_series('CPIAUCSL') # Monthly
cpi_fesl_data = fred.get_series('CPILFESL') # Monthly
cpi_inflation_data = fred.get_series('FPCPITOTLZGUSA') # Annual

In [ ]:
# Recession indicator data
rec_sahm_data = fred.get_series('SAHMREALTIME') # Monthly
rec_smooth_prob_data = fred.get_series('RECPROUSM156N') # Monthly
rec_nber_data = fred.get_series('USREC') # Monthly

In [ ]:
# Get Labor Data
unemployment_data = fred.get_series('UNRATE') # Monthly
unemployment_level_data = fred.get_series('UNEMPLOY') # Monthly
all_employed_data = fred.get_series('PAYEMS') # Monthly
employment_population_ratio_data = fred.get_series('EMRATIO') # Monthly
labor_force_participation_data = fred.get_series('CIVPART') # Monthly
job_openings_data = fred.get_series('JTS1000JOL') # Monthly
job_hires_data = fred.get_series('JTS1000HIL') # Monthly
job_separations_data = fred.get_series('JTS1000TSL') # Monthly

In [ ]:
# Get GDP Data
gdp_data = fred.get_series('GDP') # Quarterly
gdp_real_data = fred.get_series('GDPC1') # Quarterly

In [ ]:
# M1 and M2 Money Supply Data
m1_data = fred.get_series('M1SL') # Monthly
m2_data = fred.get_series('M2SL') # Monthly
m2_real_data = fred.get_series('M2REAL') # Monthly

In [ ]:
# Interest Rate Data
interest_rate_mtg_data = fred.get_series('MORTGAGE30US') # Weekly
interest_rate_fedfunds_data = fred.get_series('FEDFUNDS') # Monthly

In [ ]:
# Income Data
median_income_real_data = fred.get_series('MEHOINUSA672N') # Annual
median_income_data = fred.get_series('MEHOINUSA646N') # Annual
average_weekly_earnings_data = fred.get_series('CES0500000011') # Monthly

In [ ]:
# Consumer Sentiment Data
consumer_sentiment_data = fred.get_series('UMCSENT') # Monthly
consumer_sentiment_inflation_data = fred.get_series('MICH') # Monthly
consumer_sentiment_composite_data = fred.get_series('USACSCICP02STSAM') # Monthly
consumer_sentiment_composite_amplitude_data = fred.get_series('CSCICP03USM665S') # Monthly

In [25]:
# Create lookup tables for the meaning of the dataframes
lookup_names = [('CPIAUCSL','cpi_data')
,('CPILFESL','cpi_fesl_data')
,('FPCPITOTLZGUSA','cpi_inflation_data')
,('SAHMREALTIME','rec_sahm_data')
,('RECPROUSM156N','rec_smooth_prob_data')
,('USREC','rec_nber_data')
,('UNRATE','unemployment_data')
,('UNEMPLOY','unemployment_level_data')
,('PAYEMS','all_employed_data')
,('EMRATIO','employment_population_ratio_data')
,('CIVPART','labor_force_participation_data')
,('JTS1000JOL','job_openings_data')
,('JTS1000HIL','job_hires_data')
,('JTS1000TSL','job_separations_data')
,('GDP','gdp_data')
,('GDPC1','gdp_real_data')
,('M1SL','m1_data')
,('M2SL','m2_data')
,('M2REAL','m2real_data')
,('MORTGAGE30US','interest_rate_mtg_data')
,('FEDFUNDS','interest_rate_fedfunds_data')
,('MEHOINUSA672N','median_income_real_data')
,('MEHOINUSA646N','median_income_data')
,('CES0500000011','average_weekly_earnings_data')
,('UMCSENT','consumer_sentiment_data')
,('MICH','consumer_sentiment_inflation_data')
,('USACSCICP02STSAM','consumer_sentiment_composite_data')
,('CSCICP03USM665S','consumer_sentiment_composite_amplitude_data')] 

# Create lookup df
for i, name in enumerate(lookup_names):
    if i == 0:
        lookup_df = pd.DataFrame(fred.get_series_info(name[0])).transpose()
        lookup_df['variable_name'] = name[1]
    else:
        df = pd.DataFrame(fred.get_series_info(name[0])).transpose()
        df['variable_name'] = name[1]
        lookup_df = pd.concat([lookup_df, df])

# Dropping the real time end and observation end because we are filtering the data to have no records past on 2026-04-01 for reproducibility
lookup_df.drop(['realtime_end', 'observation_end'], axis=1, inplace=True)

# Write lookup df to file
lookup_df.to_csv(os.path.abspath(os.path.join(parent_dir, '../data/fred/fred_column_mapping_data.csv')), index=False)

In [33]:
# Get each dataframe into a list by its frequency for processing
weekly_dataframe_names = ['interest_rate_mtg_data']
weekly_dataframes = [globals()[name] for name in weekly_dataframe_names]
monthly_dataframe_names = ['cpi_data', 'cpi_fesl_data', 'rec_sahm_data', 'rec_smooth_prob_data', 'rec_nber_data', 'unemployment_data', 'unemployment_level_data', 'all_employed_data', 'employment_population_ratio_data', 'labor_force_participation_data', 'm1_data', 'm2_data', 'm2_real_data', 'interest_rate_fedfunds_data', 'consumer_sentiment_data', 'consumer_sentiment_inflation_data', 'consumer_sentiment_composite_data', 'consumer_sentiment_composite_amplitude_data', 'job_openings_data', 'job_hires_data', 'job_separations_data', 'average_weekly_earnings_data']
monthly_dataframes = [globals()[name] for name in monthly_dataframe_names]
quarterly_dataframe_names = ['gdp_data', 'gdp_real_data']
quarterly_dataframes = [globals()[name] for name in quarterly_dataframe_names]
annual_dataframe_names = ['cpi_inflation_data', 'median_income_real_data', 'median_income_data']
annual_dataframes = [globals()[name] for name in annual_dataframe_names]

In [ ]:
# Loop through weekly dataframes (series), convert to dataframe if series, and get merge columns (i.e. month, quarter, year columns)
new_weekly_dataframes = []
for i, df in enumerate(weekly_dataframes):
    if isinstance(df, pd.Series):
        df = df.to_frame().reset_index()
        df.columns = ['date', f'{weekly_dataframe_names[i]}_weekly']
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['year'] = df['date'].dt.year
        new_weekly_dataframes.append(df)

In [ ]:
# Loop through monthly dataframes (series), convert to dataframe if series, and get merge columns (i.e. month, quarter, year columns)
new_monthly_dataframes = []
for i, df in enumerate(monthly_dataframes):
    if isinstance(df, pd.Series):
        df = df.to_frame().reset_index()
        df.columns = ['date', f'{monthly_dataframe_names[i]}_monthly']
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['year'] = df['date'].dt.year
        new_monthly_dataframes.append(df)

In [ ]:
# Loop through quarterly dataframes (series), convert to dataframe if series, and get merge columns (i.e. quarter, year columns)
new_quarterly_dataframes = []
for i, df in enumerate(quarterly_dataframes):
    if isinstance(df, pd.Series):
        df = df.to_frame().reset_index()
        df.columns = ['date', f'{quarterly_dataframe_names[i]}_quarterly']
        df['quarter'] = df['date'].dt.quarter
        df['year'] = df['date'].dt.year
        new_quarterly_dataframes.append(df)

In [ ]:
# Loop through yearly  dataframes (series), convert to dataframe if series, and get merge columns (i.e. month, quarter, year columns)
new_yearly_dataframes = []
for i, df in enumerate(annual_dataframes):
    if isinstance(df, pd.Series):
        df = df.to_frame().reset_index()
        df.columns = ['date', f'{annual_dataframe_names[i]}_annual']
        df['year'] = df['date'].dt.year
        new_yearly_dataframes.append(df)

In [42]:
# Merge all the dataframes into 4 different dataframes by frequency (weekly, monthly, quarterly, annual)

# Create Monthly Dataframe
# Find the earliest date within the monthly dataframes to use as the left side of the join
for i, df in enumerate(new_monthly_dataframes):
    if i == 0:
        min_date = (monthly_dataframe_names[i], i, df['date'].min())
    elif min_date[2] > df['date'].min():
        min_date = (monthly_dataframe_names[i], i, df['date'].min())

# Earliest is ('rec_nber_data', Timestamp('1854-12-01 00:00:00'))
merged_monthly_df = new_monthly_dataframes[min_date[1]]

# Convert weekly dataframes to monthly by averaging the weekly data within each month and then merge with monthly dataframe
for df in new_weekly_dataframes:
    df_weekly_2_monthly = df.drop('date', axis=1).groupby(['year', 'quarter', 'month']).mean().reset_index()
    merged_monthly_df = pd.merge(merged_monthly_df, df_weekly_2_monthly, on=['month', 'quarter', 'year'], how='left', suffixes=['','_drop'])
# Merge monthly dataframes
for df in new_monthly_dataframes:
    merged_monthly_df = pd.merge(merged_monthly_df, df.drop('date', axis=1), on=['month', 'quarter', 'year'], how='left', suffixes=['','_drop'])
# Merge quarterly dataframes
for df in new_quarterly_dataframes:
    merged_monthly_df = pd.merge(merged_monthly_df, df.drop('date', axis=1), on=['quarter', 'year'], how='left', suffixes=['','_drop'])
# Merge annual dataframes
for df in new_yearly_dataframes:
    merged_monthly_df = pd.merge(merged_monthly_df, df.drop('date', axis=1), on=['year'], how='left', suffixes=['','_drop'])

# Drop columns with _drop suffix
merged_monthly_df = merged_monthly_df[[col for col in merged_monthly_df.columns if not col.endswith('_drop')]]

# Order columns
merged_monthly_df = merged_monthly_df[['date', 'month', 'quarter', 'year', 'interest_rate_mtg_data_weekly',
       'cpi_data_monthly', 'cpi_fesl_data_monthly', 'rec_sahm_data_monthly',
       'rec_smooth_prob_data_monthly', 'rec_nber_data_monthly',
       'unemployment_data_monthly', 'unemployment_level_data_monthly',
       'all_employed_data_monthly', 'employment_population_ratio_data_monthly',
       'labor_force_participation_data_monthly', 'm1_data_monthly',
       'm2_data_monthly', 'm2_real_data_monthly',
       'interest_rate_fedfunds_data_monthly',
       'consumer_sentiment_data_monthly',
       'consumer_sentiment_inflation_data_monthly',
       'consumer_sentiment_composite_data_monthly',
       'consumer_sentiment_composite_amplitude_data_monthly',
       'job_openings_data_monthly', 'job_hires_data_monthly',
       'job_separations_data_monthly', 'average_weekly_earnings_data_monthly',
       'gdp_data_quarterly', 'gdp_real_data_quarterly',
       'cpi_inflation_data_annual', 'median_income_real_data_annual',
       'median_income_data_annual']]

# Filter data to have no records past 2026-04-01
merged_monthly_df = merged_monthly_df[merged_monthly_df['date']<'2026-04-01']

# Write to a file
merged_monthly_df.to_csv(os.path.abspath(os.path.join(parent_dir, '../data/fred/fred_monthly_data.csv')), index=False)